In [1]:
import os
os.chdir('/container/mount/point')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import rpy2.robjects as ro
import pickle 

from tqdm.notebook import tqdm

from rpy2.robjects.packages import importr
from rpy2.robjects import pandas2ri, FloatVector
from rpy2.robjects.conversion import localconverter

from utils.helper import r_to_pandas, check_samples_overlap, generate_taxa_dict, calculate_obs_stat_and_pvalues
from utils.preprocessing import filter_and_process_asv_table

# Convert pandas.DataFrames to R dataframes automatically.
pandas2ri.activate()

In [2]:
utils = importr('utils')
devtools = importr('devtools')
linda = importr("LinDA")

In [ ]:
matched_df = pd.read_csv('data/AGP/matched_df_W.csv', sep=',', index_col=0)


### taxonomy annotation
f_taxa_smoker = pd.read_csv('data/AGP/tax_table_smoker.csv', sep=',', index_col=0)
f_taxa_non_smoker = pd.read_csv('data/AGP/tax_table_non_smoker.csv', sep=',', index_col=0)

### count data tables N x p
f_count_smoker = pd.read_csv('data/AGP/otu_table_smoker.csv', sep=',', index_col=0)
f_count_non_smoker = pd.read_csv('data/AGP/otu_table_non_smoker.csv', sep=',', index_col=0)

print("Bacterial families for both cases are the same:", (f_count_smoker.columns == f_count_non_smoker.columns).all())
family_names = f_count_smoker.columns
names_dict = {index: (f_taxa_smoker.loc[index, "Family"] if pd.notna(f_taxa_smoker.loc[index, "Family"]) else f"f_unknown_{index}") for index in family_names.map(int)}

Bacterial families for both cases are the same: True


In [ ]:
### TO DO - run NEtcomi preprocessing on your mathced_df instead of Alice's

# missing_samples = [ix for ix in f_count_non_smoker.index if ix not in matched_df[matched_df["W"] == 1].index]
# print("Missing columns:", missing_samples)

# missing_samples = [ix for ix in matched_df[matched_df["W"] == 1].index if ix not in f_count_non_smoker.index]
# print("Missing columns:", missing_samples)

Missing columns: ['10317.000028655.55074']


In [16]:
target_variable = "W"

sample_df = pd.read_csv(f'data/AGP/matched_df_{target_variable}.csv', sep=',', index_col=0)

for case, f_count in zip(["smoker", "non_smoker"], [f_count_non_smoker, f_count_smoker]):
    print(f"Processing {case} case")
    
    # Select count table 
    counts = f_count.copy().T
    p, n = counts.shape
    print(f"\n Shape(p,N): {counts.shape}")

    w = pd.DataFrame(sample_df.loc[sample_df.index.isin(counts.columns), target_variable].values, index=counts.columns, columns=[target_variable])

    # w = pd.DataFrame(df[df.index.isin(sample_ids)]["W"].values, index=sample_ids, columns=["w"])

    # lo = linda.linda(data, w, formula="~w", alpha=0.05, prev_cut=0, lib_cut=1)

    # linda_out = r_to_pandas(lo.rx2("output"))['w']

    # # observed statistic of true W            
    # stats = pd.DataFrame({
    # "obs_stat": linda_out['stat'],
    # "p_value": linda_out['pvalue'],
    # "reject_null": linda_out['reject'],
    # 'lfc': linda_out['log2FoldChange'],
    # 'lfcSE': linda_out['lfcSE'],
    # 'baseMean': linda_out['baseMean'] })
    
    # result_dict[level] = stats


counts.columns

Processing smoker case

 Shape(p,N): (40, 234)


ValueError: Shape of passed values is (233, 1), indices imply (234, 1)

In [ ]:
target_variable = "smoking_bin"
count_data_path = "data/feature_table.tsv"

sample_df = pd.read_csv(f'data/matched_df_{target_variable}.csv', sep=',', index_col=0)

### Match covariates with 16S data
asv = pd.read_csv(str(count_data_path), index_col=0, sep='\t')

p, N = asv.shape
print(p, N)

### sort ASVs by pair-mathcing order
arr = list(sample_df.index)
ASV_table = asv.reindex(sorted(asv.columns, key=lambda x: arr.index(int(x)) if x.isdigit() and int(x) in arr else float('inf')), axis=1)
ASV_table.head()

### Add taxonomic annotation
taxa = pd.read_csv('data/taxonomy_clean.csv', sep=',', index_col=0)

taxa_dict = dict()

for level in taxa.columns:
    df_level = ASV_table.join(taxa[level])
    df_level = df_level.groupby(level).sum()
    
    taxa_dict[level] = df_level
    
taxa_dict["ASVs"] = ASV_table

for level in taxa_dict.keys():
    print("{0} count table shape: {1}".format(level, taxa_dict[level].shape))

freq_threshold = 0

df, diff_samples = check_samples_overlap(sample_df, ASV_table)

str_sample_ids = set(df.index.astype(str))

ASV_table = ASV_table.loc[:, ASV_table.columns.isin(str_sample_ids)]    
asv_top99_samples, asv_samples_ids = filter_and_process_asv_table(ASV_table, freq_threshold=freq_threshold)

taxa_dict = generate_taxa_dict(asv=asv_top99_samples, taxa=taxa)
del taxa_dict["name"]
for level in taxa_dict.keys():
    print("{0} count table shape: {1}".format(level, taxa_dict[level].shape))
    
#overlap 16S and IgE samples  
sample_ids = asv_top99_samples.columns.astype(int)
w = pd.DataFrame(df[df.index.isin(sample_ids)]["W"].values, index=sample_ids, columns=["w"])

#run LinDA
linda_stats = calculate_obs_stat_and_pvalues(taxa_dict, w)
print("------------------------- LinDA is DONE ------------------------- \n")

15170 2034
domain count table shape: (2, 2034)
phylum count table shape: (23, 2034)
class count table shape: (38, 2034)
order count table shape: (140, 2034)
family count table shape: (363, 2034)
genus count table shape: (2677, 2034)
species count table shape: (13584, 2034)
name count table shape: (13670, 2034)
ASVs count table shape: (15170, 2034)
Samples that are present in matched pairs, but not in ASVs: 61
70119
59023
49292
56792
53865
51566
71272
56693
46136
58618
59549
16254
52470
91920
95682
50658
49238
52427
64895
90866
73942
93317
24095
76592
60305
14536
66420
89626
93108
24140
76424
66919
62703
47969
45566
53810
88185
11097
57411
46405
14852
21881
29452
77101
63803
88829
70082
61262
48976
72336
69750
47975
74995
60190
66890
31891
25170
95279
30477
61822
44815
These columns have not variance and will be dropped: Index(['33231'], dtype='object')
domain count table shape: (2, 442)
phylum count table shape: (15, 442)
class count table shape: (25, 442)
order count table shape: (90,

In [ ]:
w = pd.DataFrame(sample_df['W'].values, columns=['w'])

obs_stat_dict = dict()
assymptotic_pvalues = dict()

# for level in tqdm(taxa_dict.keys()):
for level in taxa_dict.keys():
    print(level)
    
    data = taxa_dict[level]
    
    p, N = data.shape
    
    print(p, N)
    
    
    if p >= 20:

        lo = linda.linda(data, w, formula="~w", alpha = 0.05, prev_cut = 0, lib_cut = 1)

        linda_out = r_to_pandas(lo.rx2("output"))
        linda_matched = linda_out['w']

        # observed dtatistic of true W
        obs_stat = linda_matched['stat']
        obs_stat_dict[level] = obs_stat

        pvalue = linda_matched['pvalue']
        assymptotic_pvalues[level] = pvalue

In [ ]:
n_iter = 10000
alpha = 0.05

# for level in taxa_dict.keys():
for level in taxa_dict.keys():

    data = taxa_dict[level]

    p, N = data.shape

    if p >= 20:

        print("Linda test for {0} level".format(level))

        test_stats = np.empty((p, 0), int)
        W_frame = sample_df.iloc[:, -n_iter:]

        for i in tqdm(range(n_iter)):

            w_tmp = W_frame.iloc[:, i].to_frame(name="w")

            # apply linda
            lo_tmp = linda.linda(data, w_tmp, formula="~w", alpha = alpha, prev_cut = 0, lib_cut = 1)

            linda_out_tmp = r_to_pandas(lo_tmp.rx2("output"))
            out = linda_out_tmp['w']

            stat = out['stat'].values.reshape(p,1)
            test_stats = np.append(test_stats, stat, axis = 1)

        test_stats = pd.DataFrame(test_stats)
        # test_stats.to_csv('data/linda_{0}_{1}_age_sex_bmi.csv'.format(level, n_iter))
        test_stats.to_csv('data/linda_{0}_{1}_ige_{2}.csv'.format(level, n_iter, dataset))

In [ ]:
n_iter = 10000

# for level in taxa_dict.keys():
for level in taxa_dict.keys():
    
    # file_path = '../data/linda_{0}_{1}_age_sex_bmi.csv'.format(level, n_iter)
    file_path = 'data/linda_{0}_{1}_ige_{2}.csv'.format(level, n_iter, dataset)

    if os.path.exists(file_path):

        print("p-value adjustment procedure for {0} level".format(level))

        data = taxa_dict[level]
        obs_stat = obs_stat_dict[level]

        test_stats = pd.read_csv(file_path, index_col=0)

        ### Step 1 - unadjusted p-values
        p_value = calculate_unadjusted_p_values(test_stats, obs_stat, test_type="two-sided")

        ### Step 2 - unadjusted randomized minimum p-values
        min_p_values = min_unadjusted_p_values(test_stats)

        ### Step 3 - adjusted p-values
        adj_p_values = adjusted_p_values(min_p_values, p_value)

        p_value["Lee et al."] = np.array(adj_p_values)

        # p_value.to_csv('../data/adj_linda_{0}_{1}_age_sex_bmi.csv'.format(level, n_iter))
        p_value.to_csv('data/adj_linda_{0}_{1}_ige_{2}.csv'.format(level, n_iter, dataset))